In [32]:
import pandas as pd

In [33]:
daily_data = pd.read_csv('../data/citibike_weather_daily.csv',
                         header=None,
                         names=["ride_date", 'num_rides','avg_duration_min', 'temp_f', 'max_temp_f','min_temp_f', 'wind_speed_knots', 'precip_in','day_of_week','month'])

In [34]:
print(daily_data.dtypes)

ride_date            object
num_rides             int64
avg_duration_min    float64
temp_f              float64
max_temp_f          float64
min_temp_f          float64
wind_speed_knots    float64
precip_in           float64
day_of_week          object
month                 int64
dtype: object


In [35]:
daily_data['ride_date'] = pd.to_datetime(daily_data['ride_date'])

numeric_cols = ['num_rides', 'avg_duration_min', 'temp_f', 'max_temp_f', 'min_temp_f', 'wind_speed_knots', 'precip_in', 'month']
for col in numeric_cols:
    daily_data[col] = pd.to_numeric(daily_data[col], errors='raise')

print(daily_data.dtypes)

ride_date           datetime64[ns]
num_rides                    int64
avg_duration_min           float64
temp_f                     float64
max_temp_f                 float64
min_temp_f                 float64
wind_speed_knots           float64
precip_in                  float64
day_of_week                 object
month                        int64
dtype: object


In [36]:
print(daily_data['ride_date'].isna().sum())
print(daily_data['ride_date'].min(), daily_data['ride_date'].max())

0
2013-07-01 00:00:00 2018-05-31 00:00:00


In [37]:
daily_data.loc[daily_data['precip_in'] == 99.99, 'precip_in'] = pd.NA
print(daily_data['precip_in'].isna().sum())

1


In [38]:
median_precip = daily_data['precip_in'].median()
daily_data['precip_in'] = daily_data['precip_in'].fillna(median_precip)
print(median_precip)

0.0


The 99.99 value in precip_in is not a real measurement as that would be extremly unrealistic for LGA, so I converted it to NaN. Only one row is affected, and the rest of that day's data (rides, temperature, wind) looks fine, so dropping the whole row would throw away good data just to avoid one bad cell. I used the colum's median precip_in value in place of the missing one as it serves as a realistic place holder; since precipitation is heavily zero inflated in this dataset (most days have 0 inches).

In [39]:
day_dummies = pd.get_dummies(daily_data['day_of_week'], prefix='day' , drop_first=True)
daily_data = pd.concat([daily_data, day_dummies], axis=1)
daily_data.head()

,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,day_of_week,month,day_Monday,day_Saturday,day_Sunday,day_Thursday,day_Tuesday,day_Wednesday
0,2013-07-01,16650,16.31,74.8,78.1,73.4,7.8,0.00,Monday,7,True,False,False,False,False,False
1,2013-07-02,22745,15.97,76.1,82.9,73.0,8.0,0.73,Tuesday,7,False,False,False,False,True,False
2,2013-07-03,21864,16.24,78.5,84.9,73.9,8.8,0.06,Wednesday,7,False,False,False,False,False,True
3,2013-07-04,22326,21.22,82.0,91.0,73.9,8.6,0.96,Thursday,7,False,False,False,True,False,False
4,2013-07-05,21842,18.04,84.4,93.0,75.9,9.0,0.00,Friday,7,False,False,False,False,False,False


I used drop_first=True when one hot encoding day_of_week. Since every row has exactly one day marked 1, keeping all 7 dummy columns would make them perfectly collinear, one column is always predictable from the other six. That's a problem for a linear regression model with an intercept, since it can make the coefficients unstable. Dropping one day (Friday, alphabetically first) makes it the implicit baseline, and every other day's coefficient gets interpreted relative to Friday instead of on its own.

In [40]:
daily_data['days_since_launch'] = (daily_data['ride_date'] - daily_data['ride_date'].min()).dt.days

I'll go with days_since_launch for modeling. Year only has 6 different values, so the model would basically have to treat every year jump as the same size effect, which doesn't seem right since ridership didn't grow the same amount every year. days_since_launch is continuous so it can pick up on the gradual growth I saw back in earlier histograms a lot smoother, plus it handles the missing 6 month in data better too.

In [41]:
daily_data['temp_f_squared'] = daily_data['temp_f'] ** 2

I added a squared temperature term because my scatter plots from E3 showed ridership isn't linear with temperature, it climbs then falls off sharply past a certain point, and a linear model can only bend into that curve if it has a squared version of the feature to work with.



In [42]:
daily_data['is_weekend'] = daily_data['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

I added an is_weekend flag because the days of week comparison from E5 showed a clear split between weekday and weekend ridership patterns, and this collapses that into one simple feature.

In [43]:
daily_data.to_csv('../data/citibike_weather_daily_clean.csv', index=False)

In [44]:
check = pd.read_csv('../data/citibike_weather_daily_clean.csv')
print(check.shape)
print(check.columns.tolist())

(1610, 19)
['ride_date', 'num_rides', 'avg_duration_min', 'temp_f', 'max_temp_f', 'min_temp_f', 'wind_speed_knots', 'precip_in', 'day_of_week', 'month', 'day_Monday', 'day_Saturday', 'day_Sunday', 'day_Thursday', 'day_Tuesday', 'day_Wednesday', 'days_since_launch', 'temp_f_squared', 'is_weekend']
